# Notebook 01: RDKit 入门 & 分子表示法

## 学习目标
- 理解 SMILES：化学分子如何用字符串表示
- 会用 RDKit 加载、显示、搜索分子
- 分子性质的核心概念

## 类比
SMILES 之于分子 = JSON 之于数据结构 = 像素数组之于图像

这是一种序列化/反序列化的思想——把三维分子结构压平成一行字符串，电脑和人都能读。

In [ ]:
# 导入 RDKit —— 开源的化学信息学工具箱
# 这就是化学界的"numpy"——所有操作分子的基础库
from rdkit import Chem
from rdkit.Chem import Draw, Descriptors
from rdkit.Chem.Draw import IPythonConsole

# 设置分子显示大小
IPythonConsole.molSize = (400, 200)

print('RDKit 版本:', Chem.rdkitVersion)

## 1. SMILES —— 分子的"序列化格式"

SMILES = Simplified Molecular Input Line Entry System

把分子写成一行字符串的规则系统。每条规则都有明确的化学含义：

| SMILES 符号 | 化学含义 | 示例 |
|------------|---------|-----|
| C | 碳原子 | C = 甲烷的骨架碳 |
| O | 氧原子 | O = 水分子 |
| N | 氮原子 | N = 氨 |
| = | 双键 | C=O = 羰基 |
| c / 小写 | 芳香环碳 | c1ccccc1 = 苯环 |
| ( ) | 支链 | CC(=O)O = 乙酸 |

**核心规则**：
- 原子用元素符号（大写字母开头）
- 单键不写（默认），双键写 `=`，三键写 `#`
- 支链用括号 `()` 括起来
- 环用数字编号标记（`C1...C1` 表示从 C1 出发，走一圈回到 C1）

In [ ]:
# 从一个简单的 SMILES 开始：水分子
water = Chem.MolFromSmiles('O')
water

In [ ]:
# 乙醇（酒精）—— C-C-O
ethanol = Chem.MolFromSmiles('CCO')
ethanol

In [ ]:
# 苯（六元碳环）—— 芳香化合物，用小写 c
benzene = Chem.MolFromSmiles('c1ccccc1')
benzene

In [ ]:
# 阿司匹林 —— 一个真正的药物分子！
# CC(=O)Oc1ccccc1C(=O)O
# 拆解：
#   CC(=O)O  -> 乙酸基团 (acetyl)
#   c1ccccc1 -> 苯环
#   C(=O)O   -> 另一个羧酸基团
aspirin = Chem.MolFromSmiles('CC(=O)Oc1ccccc1C(=O)O')
aspirin

## 2. SMILES ↔ 分子 互转

RDKit 的核心功能之一：把 SMILES 字符串变成分子对象（`MolFromSmiles`），以及反过来（`MolToSmiles`）。
这就是 **序列化（分子→字符串）** 和 **反序列化（字符串→分子）**。

In [ ]:
# SMILES → 分子对象
mol = Chem.MolFromSmiles('CC(=O)Oc1ccccc1C(=O)O')
print('分子对象:', mol)

# 分子对象 → 规范 SMILES（标准化格式）
canonical = Chem.MolToSmiles(mol)
print('规范 SMILES:', canonical)

# 注意：输入和输出可能不完全相同！
# RDKit 会重新编号原子，输出"规范"版本
# 就像 JSON.stringify 会去掉多余空格一样

## 3. 分子性质计算

这直接对应我们的 `MolPropertyCalculator`。
RDKit 内置了大量描述符（descriptors）——就是分子的"特征值"。

**类比**：这些描述符就像给一个人的"特征向量"——身高、体重、年龄——用来判断"这个人像不像运动员"。
同理，LogP、QED、TPSA 用来判断"这个分子像不像药"。

In [ ]:
# 手动计算 LogP
mol = Chem.MolFromSmiles('CC(=O)Oc1ccccc1C(=O)O')
logp = Descriptors.MolLogP(mol)
print(f'阿司匹林的 LogP = {logp:.2f}')
print(f'LogP 含义: 分子在辛醇（油）和水之间的分配系数')
print(f'  > 0: 更亲油（倾向于留在细胞膜里）')
print(f'  < 0: 更亲水（容易溶解在水中）')
print(f'  理想药物: 1-3')

In [ ]:
# TPSA —— 极性表面积
tpsa = Descriptors.TPSA(mol)
print(f'阿司匹林的 TPSA = {tpsa:.2f} Å²')
print(f'TPSA < 140 Å² → 容易穿过细胞膜')
print(f'TPSA > 140 Å² → 难以穿过细胞膜（太"极性"）')

In [ ]:
# 氢键供体/受体 —— 影响水溶性和靶点结合
hbd = Descriptors.NumHDonors(mol)   # 氢键供体（能"给"出氢键的-OH, -NH 基团）
hba = Descriptors.NumHAcceptors(mol) # 氢键受体（能"接"氢键的 O, N 原子）
mw = Descriptors.MolWt(mol)          # 分子量

print(f'氢键供体: {hbd}')
print(f'氢键受体: {hba}')
print(f'分子量: {mw:.2f} Da')
print()
print('Lipinski 五规则（口服药物的经验法则）:')
print(f'  MW < 500: {"✓" if mw < 500 else "✗"} ({mw:.0f})')
print(f'  LogP < 5: {"✓" if logp < 5 else "✗"} ({logp:.1f})')
print(f'  HBD < 5:  {"✓" if hbd < 5 else "✗"} ({hbd})')
print(f'  HBA < 10: {"✓" if hba < 10 else "✗"} ({hba})')

## 4. QED —— 药物相似性打分

QED = Quantitative Estimate of Drug-likeness

把 8 个分子性质加权求和，得出一个 0-1 的分数。
**类比**：就像信用评分——多个维度（收入、负债、还款记录）综合成一个数。
QED > 0.5 算"看起来像药"。

In [ ]:
from rdkit.Chem import QED

# 阿司匹林 vs 一个非药物分子
aspirin = Chem.MolFromSmiles('CC(=O)Oc1ccccc1C(=O)O')
# 随便一个长链烷烃（肯定不像药）
alkane = Chem.MolFromSmiles('CCCCCCCCCCCCCCCCCCCC')  # 20个碳的直链烷烃（石蜡）

print(f'阿司匹林 QED: {QED.qed(aspirin):.3f}')
print(f'长链烷烃 QED: {QED.qed(alkane):.3f}  <- 极低，因为不像药物')

## 5. ADC 连接子的真实 SMILES

现在看一个真正的 ADC 连接子。
Val-Cit-PABC 是最常用的酶裂解型连接子——被溶酶体中的组织蛋白酶 B (Cathepsin B) 切断。

In [ ]:
# Val-Cit-PABC 连接子（简化版）
# Val = 缬氨酸, Cit = 瓜氨酸, PABC = 对氨基苄氧羰基
val_cit_pabc = 'CC(C)[C@H](N)C(=O)N[C@@H](CCCNC(N)=O)C(=O)NCc1ccc(O)cc1'

linker = Chem.MolFromSmiles(val_cit_pabc)
if linker:
    print('Val-Cit-PABC 连接子:')
    display(linker)
    print(f'LogP: {Descriptors.MolLogP(linker):.2f}')
    print(f'QED: {QED.qed(linker):.3f}')
    print(f'TPSA: {Descriptors.TPSA(linker):.2f}')
    print(f'裂解机制: 酶裂解（被 Cathepsin B 在 Gly 位点切断）')
    print(f'裂解 pH: 5.0-6.0（溶酶体内）')
else:
    print('SMILES 解析失败！这不是一个有效的分子表示。')

## 6. 练习：判断 SMILES 有效性

写出几个 SMILES，判断它们是否有效。这对应我们 MCP 工具 `validate_smiles` 的底层逻辑。

In [ ]:
def test_smiles(smiles, name=''):
    """测试一个 SMILES 是否有效"""
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        print(f'❌ {name}: 无效 SMILES')
        return None
    else:
        canonical = Chem.MolToSmiles(mol)
        print(f'✅ {name}: 有效 → {canonical}')
        return mol

# 有效的 SMILES
test_smiles('CCO', '乙醇')
test_smiles('CC(=O)O', '乙酸')
test_smiles('c1ccccc1', '苯')

# 无效的 SMILES
test_smiles('XYZ', '随便写的字母')
test_smiles('C(C(C', '括号不匹配')
test_smiles('', '空字符串')

## 总结

| 概念 | 对程序员的类比 |
|------|-------------|
| SMILES | JSON/XML/CSV —— 序列化格式 |
| MolFromSmiles | `json.loads()` —— 反序列化 |
| MolToSmiles | `json.dumps()` —— 序列化（含规范化） |
| LogP | 不是一个"计算"，是一个实验测量值的模型预测 |
| QED | 信用评分 —— 多维特征加权求和 |
| RDKit | NumPy/Pandas —— 领域标准库 |

**下一步**：Week 2 会用这些能力构建 `MolPropertyCalculator` 和 `PhSimulator`。